# Language Models are Few-Shot Learners (GPT-3)

## Learning Objectives

1. Understand few-shot learning and in-context learning
2. Implement few-shot prompting for various NLP tasks
3. Compare zero-shot, one-shot, and few-shot performance
4. Use chain-of-thought prompting for reasoning tasks
5. Simulate few-shot with smaller language models

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)

print(f"Using device: {device}")

# Load a small GPT-2 model for demonstration
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
model.eval()

print(f"Model: GPT-2 (124M parameters)")

## Level 1: Zero-Shot vs. Few-Shot Prompting

Compare how models perform with and without examples in the prompt

In [ ]:
def generate_text(prompt, max_new_tokens=50, temperature=0.7):
    """Generate text conditioned on prompt"""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs['input_ids'],
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_k=50,
            pad_token_id=tokenizer.eos_token_id
        )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Zero-shot: just task description
zero_shot_prompt = "Classify sentiment: This movie is great!"
print("=== ZERO-SHOT ===")
print(f"Prompt: {zero_shot_prompt}")
result = generate_text(zero_shot_prompt)
print(f"Output: {result}\n")

# One-shot: one example
one_shot_prompt = """Classify sentiment:
Text: This movie is amazing!
Sentiment: positive

Text: This movie is great!
Sentiment:"""
print("=== ONE-SHOT ===")
print(f"Prompt: {one_shot_prompt}")
result = generate_text(one_shot_prompt, temperature=0)
print(f"Output: {result}\n")

# Few-shot: multiple examples
few_shot_prompt = """Classify sentiment:
Text: This movie is amazing!
Sentiment: positive

Text: This movie was terrible.
Sentiment: negative

Text: It was okay, nothing special.
Sentiment: neutral

Text: This movie is great!
Sentiment:"""
print("=== FEW-SHOT (3 examples) ===")
print(f"Prompt: {few_shot_prompt}")
result = generate_text(few_shot_prompt, temperature=0)
print(f"Output: {result}")

## Level 2: Few-Shot for Different NLP Tasks

Few-shot prompting works for many tasks: translation, QA, summarization, etc.

In [ ]:
# Task 1: English to French Translation
translation_prompt = """Translate English to French:
English: hello
French: bonjour

English: goodbye
French: au revoir

English: how are you?
French:"""

print("=== TRANSLATION TASK ===")
result = generate_text(translation_prompt, max_new_tokens=10, temperature=0.3)
print(f"Prompt: {translation_prompt}")
print(f"Model output: {result}\n")

# Task 2: Answer Extraction from Text
qa_prompt = """Extract the answer from the text:
Text: The capital of France is Paris.
Question: What is the capital of France?
Answer: Paris

Text: Albert Einstein was born in 1879.
Question: When was Albert Einstein born?
Answer: 1879

Text: The Great Wall of China is about 13,000 miles long.
Question: How long is the Great Wall of China?
Answer:"""

print("=== QUESTION ANSWERING ===")
result = generate_text(qa_prompt, max_new_tokens=10, temperature=0.1)
print(f"Model output: {result}\n")

# Task 3: Text Summarization
summary_prompt = """Summarize the text in one sentence:
Text: The cat was sleeping under the tree. It had been a long day.
Summary: The cat was resting after a tiring day.

Text: John went to the store to buy milk. He also bought bread and eggs.
Summary: John shopped for milk, bread, and eggs.

Text: The weather is beautiful today. The sun is shining and there are no clouds.
Summary:"""

print("=== SUMMARIZATION ===")
result = generate_text(summary_prompt, max_new_tokens=20, temperature=0.3)
print(f"Model output: {result}")

## Real-World Example 1: Chain-of-Thought Prompting

Ask the model to think step-by-step for reasoning tasks

In [ ]:
# Standard few-shot (no reasoning steps)
direct_prompt = """Q: If there are 3 cats and each has 4 legs, how many legs total?
A: 12

Q: If there are 5 dogs and each has 4 legs, how many legs total?
A: 20

Q: If there are 7 birds and each has 2 wings, how many wings total?
A:"""

print("=== DIRECT ANSWER (no reasoning) ===")
print(f"Prompt: {direct_prompt}")
result = generate_text(direct_prompt, max_new_tokens=5, temperature=0)
print(f"Output: {result}\n")

# Chain-of-thought: show reasoning steps
cot_prompt = """Q: If there are 3 cats and each has 4 legs, how many legs total?
A: Let me think. Each cat has 4 legs. 3 cats total. 3 * 4 = 12 legs.

Q: If there are 5 dogs and each has 4 legs, how many legs total?
A: Each dog has 4 legs. 5 dogs total. 5 * 4 = 20 legs.

Q: If there are 7 birds and each has 2 wings, how many wings total?
A: Let me think. Each bird has 2 wings."""

print("=== CHAIN-OF-THOUGHT (with reasoning) ===")
print(f"Prompt: {cot_prompt}")
result = generate_text(cot_prompt, max_new_tokens=20, temperature=0.3)
print(f"Output: {result}")

## Real-World Example 2: Prompt Engineering for Quality

Small changes in prompt can significantly affect output quality

In [ ]:
# Vague prompt
vague_prompt = """Classify: text

This movie is great"""

print("=== VAGUE PROMPT ===")
print(f"Prompt: {vague_prompt}")
result = generate_text(vague_prompt, max_new_tokens=30, temperature=0.3)
print(f"Output: {result}\n")

# Clear prompt with formatting
clear_prompt = """Classify the movie review as positive (1) or negative (0).
Output only the label, nothing else.

Example:
Review: This movie is amazing!
Label: 1

Example:
Review: Terrible film, waste of time.
Label: 0

Review: This movie is great
Label:"""

print("=== CLEAR, WELL-FORMATTED PROMPT ===")
print(f"Prompt: {clear_prompt}")
result = generate_text(clear_prompt, max_new_tokens=5, temperature=0)
print(f"Output: {result}")

## Real-World Example 3: Few-Shot Scaling and Model Size Effects

Larger models benefit more from few-shot examples

In [ ]:
# Simulate different model sizes and their few-shot performance
# In reality: GPT-2 (124M) < GPT-3 (175B)

model_sizes = ['Small (125M)', 'Medium (1.3B)', 'Large (6.7B)', 'XLarge (175B)']
zero_shot_acc = [0.60, 0.65, 0.72, 0.82]  # Zero-shot accuracy by model size
few_shot_acc = [0.65, 0.75, 0.85, 0.95]   # Few-shot (5 examples) accuracy

x = np.arange(len(model_sizes))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, zero_shot_acc, width, label='Zero-shot')
bars2 = ax.bar(x + width/2, few_shot_acc, width, label='Few-shot (5 examples)')

ax.set_ylabel('Accuracy')
ax.set_title('Few-Shot Learning Improves with Model Scale')
ax.set_xticks(x)
ax.set_xticklabels(model_sizes)
ax.legend()
ax.set_ylim([0.5, 1.0])

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}',
                ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("Key insight: Larger models benefit MORE from few-shot examples.")
print("GPT-3 (175B) gains 13% from few-shot, while small models gain only 5%.")

## Key Takeaways

**Few-Shot Learning:**
- Zero-shot: Just task description, no examples
- One-shot: One input-output example
- Few-shot: 3-5 examples (often optimal)
- More examples can hurt if they're similar (diminishing returns)

**In-Context Learning:**
- Model learns from examples in the prompt without parameter updates
- Emergent ability: appears mainly in 175B+ models
- Works best for tasks similar to pre-training data

**Chain-of-Thought:**
- Ask models to think step-by-step
- Improves reasoning tasks (math, logic)
- Costs more tokens but often worth it

**Prompt Engineering:**
- Clear, specific prompts > vague prompts
- Consistent formatting in examples
- Specify output format (JSON, label only, etc.)

**When to use Few-Shot vs Fine-Tuning:**
- Few-shot: quick iteration, many tasks, small labeled data
- Fine-tuning: maximum accuracy, consistent task, 1K+ labeled examples
- Hybrid: few-shot + fine-tuning for best results